# 构建预测系列因子

In [103]:
import json  
import os
import re
import polars as pl
from pathlib import Path
import warnings

In [104]:
TASK_ID_PREFIX = 'basepre1'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE = True # 是否保存数据
EXCLUDE_FIRST_N = 4 # 排除最早的n个任务，因为还比较欠拟合
os.makedirs(SAVE_BASE_DIR, exist_ok=True)

In [105]:
matching_task_ids = [task_id for task_id in os.listdir(RESULTS_BASE_DIR) if pattern.match(task_id)] # 匹配中缀

In [106]:
# 列出task_id下的所有no
pattern = re.compile(r'\d{8}_\d{4}_' + TASK_ID_PREFIX + r'_[a-z0-9\-]+')
matching_task_ids = [task_id for task_id in os.listdir(RESULTS_BASE_DIR) if pattern.match(task_id)] # 匹配中缀 
matching_task_ids.sort()
matching_task_ids = matching_task_ids[EXCLUDE_FIRST_N:]

# 获取其下所有node的路径
node_paths = [] # 所有该task_id中缀的node路径
for task_id in matching_task_ids:
    node_paths.extend(
        os.path.join(
            RESULTS_BASE_DIR, task_id, 
            node_path
        )
        for node_path in os.listdir(os.path.join(RESULTS_BASE_DIR, task_id))
    ) 

len(node_paths)


90

In [107]:
# 获取其下所有performance_and_record_*.jsonl文件的路径
jsonl_paths = []
for node_path in node_paths:
    all_files = os.listdir(node_path)
    perf_and_rewa_jsonl_files = [file for file in all_files if file.endswith('.jsonl') and 'performance_and_reward_' in file]
    jsonl_paths.extend(
        os.path.join(node_path, file)
        for file in perf_and_rewa_jsonl_files
    )
jsonl_paths = [p for p in jsonl_paths if os.path.getsize(p) > 0]
len(jsonl_paths)

525

In [108]:
lazy_frames = [
    pl.scan_ndjson(f) \
        .with_columns(
            pl.col('stock').str.replace_all(':','').alias('stock'),
            pl.col(['year','month']).cast(pl.Int64)
        ) 
        
    for f in jsonl_paths
]
lf = pl.concat(lazy_frames)   # 得到 LazyFrame


In [109]:
lf = lf.select(
    pl.date(pl.col('year'),pl.col('month'),pl.lit(1)).alias('date'),
    pl.col('stock').alias('portfolio'),
    pl.col('real_return').alias('return'),
    pl.col('predict_return')
)

lf = lf.group_by(
    pl.col(['date','portfolio'])
).agg(
    pl.col('return').first().alias('return'),
    pl.col('predict_return').mean().alias('MA')
)



In [110]:
if SAVE:
    df = lf.collect(engine='gpu')
    df.write_parquet(f'{SAVE_BASE_DIR}/MA因子.parquet')
    df.write_parquet(f'{SAVE_BASE_DIR}/MA因子_copy.parquet')
